# Kvantifikace sdíleného dědictví návrhu ve flotile výkonových transformátorů pomocí PROC INBREED

## Shrnutí

Transformátory rozvodny provozovatele přenosové a distribuční soustavy jsou navrhovány v postupných
generacích, přičemž každý nový model vychází ze dvou předchozích návrhů. Tento sešit nahlíží na tuto
inženýrskou genealogii jako na rodokmen a pomocí **PROC INBREED** počítá koeficienty inbreedingu a
aditivní příbuznosti (kovariance), čímž kvantifikuje, kolik návrhového dědictví sdílejí libovolná dvě
zařízení.

Rodokmen je sestaven tak, aby byla struktura interpretovatelná: dva ze čtyř současných modelů flotily
pocházejí z dvojice **sourozeneckých** předchozích návrhů, a nesou tedy koncentrované dědictví, zatímco
ostatní vycházejí z nesouvisejících linií. Procedura toto přesně odhalí. Oba sourozenecky odvozené
modely (`G3_T1`, `G3_T3`) mají koeficient inbreedingu **F = 0,25**; zbylé dva (`G3_T2`, `G3_T4`) mají
**F = 0**. Průměrný koeficient inbreedingu flotily je **0,0417**. Při čtení matice aditivní příbuznosti
je nejméně příbuzným párem současné flotily **`G3_T1` a `G3_T3` (příbuznost = 0)** — nesdílejí žádné
společné předky a tvoří nejsilnější redundantní (N-1) pár, což je významné, protože `G3_T1` je sám o
sobě jedním z nejvíce inbredních návrhů.

## Zdroje dat

| Dataset | Řádky | Klíčové proměnné | Popis |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | Malý, deterministický inženýrský rodokmen návrhů transformátorů rozvodny napříč třemi nepřekrývajícími se generacemi návrhu (4 zakládající platformy, 4 odvozeniny druhé generace, 4 modely současné flotily). Každý nezakládající návrh zaznamenává dva odlišné předchozí návrhy (`Parent1`, `Parent2`), z nichž byl odvozen. `Sex` označuje vedoucí návrhovou linii (M = mechanická/jádrová linie, F = elektrická/vinutí). Rodokmen je zadán ručně — nikoli náhodně — takže struktura příbuznosti je předem známa a lze ji ověřit proti výstupu procedury. |

# Kvantifikace sdíleného dědictví návrhu ve flotile výkonových transformátorů

## Proč se energetická společnost zajímá o „inbreeding“

Provozovatel přenosové a distribuční soustavy provozuje stovky výkonových transformátorů rozvoden. Aby
výrobci udrželi náklady na vývoj a kvalifikační úsilí pod kontrolou, jen zřídka navrhují každý
transformátor od nuly — každý nový model **dědí** základní geometrii, topologii vinutí, izolační
systémy a konstrukci průchodek od jednoho nebo dvou předchozích modelů. Během několika nákupních
cyklů tak vzniká skutečná *inženýrská genealogie*: rodokmen návrhů.

Toto sdílené dědictví představuje skrytou hrozbu pro spolehlivost. Pokud dva transformátory chránící
stejnou zátěž (redundantní pár **N-1**) pocházejí ze stejného předchozího návrhu, je pravděpodobné, že
se skrytá konstrukční vada — rezonanční mód, mechanismus stárnutí izolace, cesta přeskoku na
průchodce — vyskytuje v **obou** jednotkách. Jediná kořenová příčina pak může vyřadit redundantní pár
současně: *společná porucha (common-mode failure)*.

**PROC INBREED** byl vytvořen právě pro kvantifikaci tohoto typu sdíleného původu. Ačkoliv má svůj
původ v chovu zvířat a rostlin, jeho matematika — Wrightova rekurze aditivní příbuznosti — je nezávislá
na oboru: měří podíl návrhového dědictví, který sdílejí dva jedinci prostřednictvím společných předků.
Slovník genetiky mapujeme na inženýrství aktiv:

| Koncept INBREED | Interpretace pro energetiku |
|---|---|
| Jedinec | Návrh / model transformátoru |
| Rodič (otec/matka) | Předchozí návrh, ze kterého byl odvozen |
| Generace (CLASS) | Nákupní / návrhový cyklus |
| Koeficient inbreedingu *F* | Míra vlastního (sebe-podobného) dědictví v rámci návrhu |
| Koeficient kovariance / příbuznosti | Sdílené návrhové dědictví mezi dvěma návrhy |
| Nejméně příbuzný pár | Nejlepší redundantní pár pro odolnost N-1 |

## Krok 1 — Zadání rodokmenu návrhu

Dataset `DesignLineage` zadáváme přímo, aby byla struktura příbuznosti jednoznačná. Existují tři
**nepřekrývající se generace návrhu**:

- **Generace 1** — čtyři zakládající platformové návrhy (`G1_P1`-`G1_P4`) bez předchůdců (prázdní
  rodiče).
- **Generace 2** — čtyři odvozené návrhy (`G2_D1`-`G2_D4`), z nichž každý vychází ze dvou **odlišných**
  platforem generace 1. `G2_D1` a `G2_D2` jsou *plnorodí sourozenci* (oba z `G1_P1` x `G1_P2`);
  `G2_D3` a `G2_D4` jsou plnorodí sourozenci (oba z `G1_P3` x `G1_P4`).
- **Generace 3** — čtyři modely současné flotily (`G3_T1`-`G3_T4`). `G3_T1` je postaven na
  sourozeneckém páru `G2_D1` x `G2_D2` a `G3_T3` na sourozeneckém páru `G2_D3` x `G2_D4`; tyto dva
  návrhy proto koncentrují dědictví. `G3_T2` a `G3_T4` každý kombinují obě nesouvisející linie.

Sloupce `ID`, `Parent1` a `Parent2` nesou rodokmen; `Sex` zaznamenává, která inženýrská linie návrh
vedla. Krátký navazující DATA krok vyprázdní zástupný symbol `.`, aby zakládající platformy měly
prázdné rodiče, jak PROC INBREED očekává.

In [1]:
/* --------------------------------------------------------
   Zadání rodokmenu návrhu transformátoru
   -------------------------------------------------------- */
data DesignLineage;
   DÉLKA ID $12 Parent1 $12 Parent2 $12 Sex $1;
   ŠTÍTEK Generation='Generace' ID='Označení' Parent1='Rodič 1' Parent2='Rodič 2' Sex='Vedoucí linie';
   INFILE DATALINES truncover;
   VSTUP Generation ID $ Parent1 $ Parent2 $ Sex $;
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
SPUSTIT;

/* Zakládající platformy nemají předchůdce: vyprázdnit zástupné tečky */
data DesignLineage;
   NASTAVIT DesignLineage;
   KDYŽ Parent1 = '.' PAK Parent1 = ' ';
   KDYŽ Parent2 = '.' PAK Parent2 = ' ';
SPUSTIT;

NÁZEV 'Rodokmen návrhu transformátoru';
PROCEDURA TISK data=DesignLineage noobs ŠTÍTEK;
   PROMĚNNÁ Generation ID Parent1 Parent2 Sex;
SPUSTIT;

                                             Rodokmen návrhu transformátoru                                             


Generace    Označení   Rodič 1   Rodič 2   Vedoucí linie
--------  ----------  --------  --------  --------------
       1  G1_P1                           M
       1  G1_P2                           M
       1  G1_P3                           M
       1  G1_P4                           F
       2  G2_D1       G1_P1     G1_P2     M
       2  G2_D2       G1_P1     G1_P2     F
       2  G2_D3       G1_P3     G1_P4     F
       2  G2_D4       G1_P3     G1_P4     M
       3  G3_T1       G2_D1     G2_D2     M
       3  G3_T2       G2_D1     G2_D3     F
       3  G3_T3       G2_D3     G2_D4     F
       3  G3_T4       G2_D2     G2_D4     M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Rodokmen návrhu transformátoru.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Krok 2 — Koeficienty inbreedingu podle generace návrhu

PROC INBREED spouštíme v režimu **více generací** uvedením `Generation` ve výkazu `CLASS`, což
vytiskne souhrn rodokmenu podle generace. Výkaz `VAR` dodává tři sloupce rodokmenu v požadovaném
pořadí: jedinec, první předchůdce, druhý předchůdce.

- Volba **IND** vytiskne koeficient inbreedingu *F* každého návrhu — míru toho, jak koncentrované je
  jeho vlastní dědictví. Návrh postavený na dvou blízce příbuzných předchůdcích nese vysoké *F*.
- Volba **MATRIX** vytiskne úplnou matici aditivní příbuznosti, takže lze párové dědictví číst přímo.
- Volba **AVERAGE** připojí celoflotilový průměrný koeficient inbreedingu — jediný souhrnný ukazatel
  diverzity návrhů.

Hodnoty blízké 0 znamenají nezávislé linie; *F* roste s tím, jak jsou si dva předchůdci návrhu blíže
příbuzní.

In [2]:
NÁZEV 'Koeficienty inbreedingu podle generace návrhu';
PROCEDURA inbreed data=DesignLineage ind average MATRIX;
   PROMĚNNÁ ID Parent1 Parent2;
   TŘÍDA Generation;
SPUSTIT;

                                     Koeficienty inbreedingu podle generace návrhu                                      


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generace/Class
--------------------------------------
Generace 1            Members = 4
Generace 2            Members = 4
Generace 3            Members = 4

Inbreeding Coefficients (Generace 1)
--------------------------------------
Označení            F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generace 2)
--------------------------------------
Označení            F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (G


NOTE: Option TITLE changed to Koeficienty inbreedingu podle generace návrhu.
NOTE: PROC INBREED data=DesignLineage



## Krok 3 — Koeficienty kovariance uložené pro navazující hodnocení rizika

Nahrazením pohledu na inbreeding volbou **COVAR** procedura vykáže stejné vztahy jako **koeficienty
kovariance (aditivní příbuznosti)** — formu, kterou by tým správy aktiv využil pro skóre rizika
redundance. Volba **OUTCOV=** zachytí úplnou matici jako dataset (`DesignCovar`), kde sloupce
`Col1`-`Col12` obsahují příbuznost každého návrhu ke každému jinému návrhu (v pořadí rodokmenu) —
připraveno k připojení zpět k přiřazení rozvoden.

Výstupní dataset vypisujeme, aby byla uložená matice viditelná vedle výpisu procedury.

In [3]:
NÁZEV 'Koeficienty kovariance (příbuznosti)';
PROCEDURA inbreed data=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   PROMĚNNÁ ID Parent1 Parent2;
   TŘÍDA Generation;
SPUSTIT;

NÁZEV 'Matice příbuznosti z OUTCOV= uložená pro hodnocení rizika';
PROCEDURA TISK data=DesignCovar noobs;
SPUSTIT;

                                          Koeficienty kovariance (příbuznosti)                                          


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generace/Class
--------------------------------------
Generace 1            Members = 4
Generace 2            Members = 4
Generace 3            Members = 4

Inbreeding Coefficients (Generace 1)
--------------------------------------
Označení            F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generace 1)
--------------------------------------------------
Označení               G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.0000
G1_P4    


NOTE: Option TITLE changed to Koeficienty kovariance (příbuznosti).
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to Matice příbuznosti z OUTCOV= uložená pro hodnocení rizika.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Krok 4 — Nejméně příbuzné páry pro redundantní (N-1) instalace

Uložená matice `DesignCovar` je přesně to, co studie redundance potřebuje. Příbuznost návrhu *i* k
návrhu *j* se nachází v řádku *i*, sloupci `Col`*j* (sloupce jsou v pořadí rodokmenu, takže
`Col9`-`Col12` odpovídají čtyřem modelům současné flotily `G3_T1`-`G3_T4`).

Ze `DesignCovar` načteme čtyři řádky současné flotily, vytvoříme každý neuspořádaný pár současné
flotily, připojíme jeho koeficient příbuznosti a seřadíme od nejméně příbuzných. Cílem je zvolit
redundantní páry, jejichž sdílené dědictví je **nejnižší** — ty minimalizují pravděpodobnost, že jedna
zděděná konstrukční vada vyřadí obě poloviny pozice chráněné N-1.

In [4]:
/* Načíst čtyři řádky současné flotily; Col9..Col12 jsou příbuznosti
   k G3_T1..G3_T4 (pořadí sloupců podle rodokmenu). Sestavit každý
   neuspořádaný pár. */
data Pairs;
   NASTAVIT DesignCovar;
   KDE ID IN ('G3_T1','G3_T2','G3_T3','G3_T4');
   DÉLKA DesignA $12 DesignB $12;
   ŠTÍTEK DesignA='Návrh A' DesignB='Návrh B' Relationship='Příbuznost';
   POLE g3{4} Col9 Col10 Col11 Col12;
   POLE nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   OPAKUJ j = 1 TO 4;
      DesignB = nm{j};
      KDYŽ DesignA < DesignB PAK OPAKUJ;
         Relationship = g3{j};
         VÝSTUP;
      KONEC;
   KONEC;
   PONECHAT DesignA DesignB Relationship;
SPUSTIT;

PROCEDURA ŘADIT data=Pairs; PODLE Relationship; SPUSTIT;

NÁZEV 'Kandidátní redundantní (N-1) páry, od nejméně příbuzných';
PROCEDURA TISK data=Pairs noobs ŠTÍTEK;
   PROMĚNNÁ DesignA DesignB Relationship;
SPUSTIT;
NÁZEV;

                                Kandidátní redundantní (N-1) páry, od nejméně příbuzných                                


 Návrh A   Návrh B    Příbuznost
--------  --------  ------------
G3_T1     G3_T3                0
G3_T2     G3_T4             0.25
G3_T1     G3_T2            0.375
G3_T1     G3_T4            0.375
G3_T2     G3_T3            0.375
G3_T3     G3_T4            0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Kandidátní redundantní (N-1) páry, od nejméně příbuzných.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Interpretace výsledků

**Koeficienty inbreedingu (krok 2).** Všechny zakládající platformy generace 1 i všechny odvozeniny
generace 2 vykazují *F* = 0 — podle konstrukce nemá žádná z nich dva příbuzné předchůdce. Dva modely
generace 3 vykazují *F* = 0,25: `G3_T1` (postaven na sourozeneckém páru `G2_D1` x `G2_D2`) a `G3_T3`
(na sourozeneckém páru `G2_D3` x `G2_D4`). Jejich předchůdci se sbíhají ke *stejné* dvojici
zakládajících platforem, takže jejich dědictví je koncentrované. Z hlediska spolehlivosti jde o
návrhy nejvíce vystavené jediné zděděné vadě a zasluhují dodatečné nezávislé kvalifikační testování.
`G3_T2` a `G3_T4`, které kombinují obě nesouvisející linie, mají *F* = 0.

**Matice příbuznosti (krok 3).** Mimodiagonální prvky kvantifikují párové sdílené dědictví. Oba
sourozenecké páry generace 2 vykazují vzájemnou příbuznost 0,50 (`G2_D1`-`G2_D2` a `G2_D3`-`G2_D4`),
zatímco odvozeniny z opačných linií mají 0,00. Inbrední modely současné flotily nesou vlastní
příbuznost 1,25 (= 1 + *F*), viditelnou na diagonále. Dataset `DesignCovar` zpřístupňuje úplnou matici
pro připojení k přiřazení rozvoden.

**Nejméně příbuzné páry (krok 4).** Seřazení všech párů současné flotily podle příbuznosti klade
**`G3_T1` a `G3_T3` na první místo s příbuzností 0,00** — pocházejí ze zcela nesouvisejících
předchozích platforem a nesdílejí žádné návrhové dědictví. Jde o nejsilnější redundantní pár, a je
obzvlášť cenný, protože `G3_T1` je sám o sobě jedním ze dvou nejvíce inbredních návrhů: spárování s
naprosto nepříbuzným protějškem zajišťuje proti jeho koncentrovanému dědictví. Druhý nejlepší pár je
`G3_T2` a `G3_T4` s příbuzností 0,25; zbývající páry mají 0,375. Průměrný koeficient inbreedingu
flotily **0,0417** (vytištěný volbou AVERAGE v kroku 2) shrnuje celkovou diverzitu návrhů. Nákup by
měl upřednostňovat páry s nejnižší příbuzností pro pozice N-1 a postupně rozšiřovat základnu předků —
inženýrský ekvivalent vyhýbání se genetickému hrdlu láhve.

**Upozornění.** Jde o ilustrativní syntetická data; produkční studie by rodokmen sestavila ze
skutečných záznamů revizí návrhu výrobce a ověřila skóre příbuznosti proti historickým událostem
společné poruchy, než by na jejich základě rozhodovala o nákupu.